In [0]:
# ============================================================
# 03_delta_lifecycle.py
# Phase 2 — Delta Lifecycle: OPTIMIZE, Time Travel, VACUUM
# ============================================================

import sys
sys.path.append("/Workspace/Users/jaswanth.vudduru@gmail.com/taxi-medallion-pipeline")
from config import BRONZE_PATH, LIFECYCLE_DEMO_PATH
from utils.schemas import bronze_schema

# -----------------------------------------------------------
# STEP 1: Simulate many small writes (what happens in real ingestion)
# This creates the "small files problem" OPTIMIZE solves
# -----------------------------------------------------------
for i in range(5):
    small_batch = spark.read.format("delta").load(BRONZE_PATH).limit(2)
    small_batch.write.format("delta").mode("append").save(LIFECYCLE_DEMO_PATH)

# Check how many files exist BEFORE optimize
files_before = dbutils.fs.ls(f"{LIFECYCLE_DEMO_PATH}/")
print(f"Files BEFORE OPTIMIZE: {len([f for f in files_before if f.name.endswith('.parquet')])}")

# Check history so far
spark.sql(f"DESCRIBE HISTORY delta.`{LIFECYCLE_DEMO_PATH}`").show(truncate=False)


In [0]:
# -----------------------------------------------------------
# STEP 2: OPTIMIZE — bin-packing small files into larger ones
# -----------------------------------------------------------
from config import LIFECYCLE_DEMO_PATH

spark.sql(f"OPTIMIZE delta.`{LIFECYCLE_DEMO_PATH}`")

files_after = dbutils.fs.ls(LIFECYCLE_DEMO_PATH)
print(f"Files AFTER OPTIMIZE: {len([f for f in files_after if f.name.endswith('.parquet')])}")
# You'll see fewer, larger files — this is bin-packing



In [0]:
# -----------------------------------------------------------
# STEP 3: Time Travel — query specific versions
# -----------------------------------------------------------
from config import LIFECYCLE_DEMO_PATH

print("\n--- FULL VERSION HISTORY ---")
spark.sql(f"DESCRIBE HISTORY delta.`{LIFECYCLE_DEMO_PATH}`").show(truncate=False)

# Query version 0 (very first write)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(LIFECYCLE_DEMO_PATH)
print(f"Version 0 row count: {v0.count()}")

# Query version 2 (after 3rd append)
v2 = spark.read.format("delta").option("versionAsOf", 2).load(LIFECYCLE_DEMO_PATH)
print(f"Version 2 row count: {v2.count()}")

# Query latest
latest = spark.read.format("delta").load(LIFECYCLE_DEMO_PATH)
print(f"Latest version row count: {latest.count()}")

# How time travel works internally:
# Delta reads the _delta_log up to the requested version number,
# replays only the add/remove entries up to that point,
# and ignores everything after — rebuilding exactly that snapshot.



In [0]:
# -----------------------------------------------------------
# STEP 4: VACUUM — purge old parquet files
# -----------------------------------------------------------
from config import LIFECYCLE_DEMO_PATH

# Before VACUUM: old parquet files still exist on disk (for time travel)
print("\nFiles in _delta_log before VACUUM:")
display(dbutils.fs.ls(f"{LIFECYCLE_DEMO_PATH}/_delta_log/"))

# VACUUM removes parquet files no longer referenced by ANY version
# within the retention window (default = 7 days)
# Note: RETAIN 0 HOURS requires setting retentionDurationCheck, which is not available on serverless
# Using default retention (7 days) instead
spark.sql(f"VACUUM delta.`{LIFECYCLE_DEMO_PATH}`")

# After VACUUM: old unreferenced files are gone
# Time travel to old versions no longer works (files deleted)
print("\n✅ VACUUM complete — old parquet files purged")
print("⚠️  Time travel to purged versions now fails — this is expected")